# M1 Notebook 11 — Automatic Differentiation and Gradient Checking

**Notebook ID:** M1_N11  
**Status:** Runnable first edition  
**Random seed:** 42

> Automatic differentiation applies the chain rule exactly through a computational graph. Gradient checking compares that result with an independent numerical approximation.


## 1. Learning objectives

1. Distinguish symbolic, numerical, and automatic differentiation.
2. Understand dual numbers and forward-mode differentiation.
3. Apply the chain rule through computation.
4. Compute multivariate gradients with seeded dual numbers.
5. Perform finite-difference gradient checking.
6. Diagnose incorrect analytical gradients.
7. Connect automatic differentiation to neural networks and Decision Intelligence.


In [ ]:
from srai_math.utils import environment_info, set_seed
from srai_math.calculus import (
    Dual,
    autodiff_derivative,
    autodiff_gradient,
    derivative,
    dual_cos,
    dual_exp,
    dual_log,
    dual_sin,
    gradient,
    gradient_check,
    relative_gradient_error,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()


## 2. Three differentiation paradigms

### Symbolic differentiation

Transforms an expression into another expression.

### Numerical differentiation

Approximates derivatives using nearby function values.

### Automatic differentiation

Evaluates exact chain-rule derivatives alongside the original computation, up to floating-point arithmetic.


## 3. Dual numbers

A dual number has the form

\[
x+\varepsilon x',
\qquad
\varepsilon^2=0.
\]

For a smooth function,

\[
f(x+\varepsilon)
=
f(x)+\varepsilon f'(x).
\]

The real component carries the function value; the dual component carries the derivative.


In [ ]:
x = Dual(3.0, 1.0)
y = x*x + 2*x + 1

assert np.isclose(y.value, 16.0)
assert np.isclose(y.derivative, 8.0)

y


The computation above automatically applies the product rule and sum rule.


## 4. Elementary functions

For example,

\[
\frac{d}{dx}\sin x=\cos x,
\qquad
\frac{d}{dx}e^x=e^x.
\]


In [ ]:
f = lambda x: dual_sin(x) * dual_exp(x)

value, derivative_ad = autodiff_derivative(f, 0.5)
derivative_exact = np.exp(0.5) * (
    np.sin(0.5) + np.cos(0.5)
)

assert np.isclose(derivative_ad, derivative_exact, rtol=1e-12)

{
    "function_value": value,
    "autodiff_derivative": derivative_ad,
    "exact_derivative": derivative_exact,
}


## 5. Compare automatic and numerical differentiation

In [ ]:
scalar_function = lambda x: np.sin(x) * np.exp(x)
x0 = 0.5

finite_difference = derivative(
    scalar_function,
    x0,
    h=1e-5,
    method="central",
)

comparison = pd.DataFrame({
    "method": ["Automatic differentiation", "Central finite difference", "Exact"],
    "derivative": [derivative_ad, finite_difference, derivative_exact],
})
comparison["absolute_error"] = np.abs(
    comparison["derivative"] - derivative_exact
)
comparison


Automatic differentiation avoids truncation error from finite differences, although both methods still use floating-point arithmetic.


## 6. Computational graphs and chain rule

Consider

\[
z=(x^2+1)e^x.
\]

The computation can be decomposed into intermediate nodes:

\[
u=x^2,\quad
v=u+1,\quad
w=e^x,\quad
z=vw.
\]

Automatic differentiation propagates derivatives through this graph.


In [ ]:
graph_function = lambda x: (x*x + 1) * dual_exp(x)
graph_value, graph_derivative = autodiff_derivative(
    graph_function,
    1.0,
)

graph_expected = np.exp(1.0) * (1.0**2 + 2*1.0 + 1.0)

assert np.isclose(graph_derivative, graph_expected)

graph_value, graph_derivative


## 7. Multivariate forward-mode gradient

For

\[
f(x,y)=x^2+3y^2+2xy,
\]

\[
\nabla f(x,y)
=
\begin{bmatrix}
2x+2y\\
6y+2x
\end{bmatrix}.
\]


In [ ]:
def multivariate_function(xs):
    x, y = xs
    return x*x + 3*y*y + 2*x*y

point = np.array([1.0, 2.0])
value, gradient_ad = autodiff_gradient(
    multivariate_function,
    point,
)
gradient_exact = np.array([6.0, 14.0])

assert np.allclose(gradient_ad, gradient_exact)

value, gradient_ad


## 8. Gradient checking with finite differences

In [ ]:
numeric_function = lambda x: (
    x[0]**2
    + 3*x[1]**2
    + 2*x[0]*x[1]
)

gradient_fd = gradient(
    numeric_function,
    point,
    h=1e-5,
)

error = relative_gradient_error(
    gradient_ad,
    gradient_fd,
)

assert gradient_check(
    gradient_ad,
    gradient_fd,
    tolerance=1e-6,
)

{
    "autodiff_gradient": gradient_ad,
    "finite_difference_gradient": gradient_fd,
    "relative_error": error,
}


A common symmetric relative-error measure is

\[
\frac{\|g_{\text{analytic}}-g_{\text{numeric}}\|}
{\|g_{\text{analytic}}\|+\|g_{\text{numeric}}\|}.
\]


## 9. Detecting an incorrect gradient

In [ ]:
incorrect_gradient = np.array([6.0, 12.0])

correct_error = relative_gradient_error(
    gradient_ad,
    gradient_fd,
)
incorrect_error = relative_gradient_error(
    incorrect_gradient,
    gradient_fd,
)

pd.DataFrame({
    "candidate": ["Correct autodiff gradient", "Incorrect gradient"],
    "relative_error": [correct_error, incorrect_error],
    "passes_check": [
        gradient_check(gradient_ad, gradient_fd),
        gradient_check(incorrect_gradient, gradient_fd),
    ],
})


## 10. Step-size sensitivity in gradient checking

In [ ]:
step_sizes = 10.0 ** (-np.arange(1, 13))
errors = []

for h in step_sizes:
    g_fd = gradient(numeric_function, point, h=h)
    errors.append(relative_gradient_error(gradient_ad, g_fd))

step_results = pd.DataFrame({
    "h": step_sizes,
    "relative_gradient_error": errors,
})
step_results


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(
    step_results["h"],
    step_results["relative_gradient_error"],
)
ax.set_xlabel("Finite-difference step size")
ax.set_ylabel("Relative gradient error")
ax.set_title("Gradient-Check Error versus Step Size")
plt.show()


The U-shaped pattern reflects a trade-off between truncation error at large steps and floating-point cancellation at very small steps.


## 11. Forward mode versus reverse mode

Forward mode propagates derivatives from inputs to outputs. Its cost scales naturally with the number of input directions.

Reverse mode propagates sensitivities from outputs backward to inputs. It is especially efficient when a scalar loss depends on many parameters, which is why backpropagation uses reverse-mode automatic differentiation.


## 12. AI interpretation

Automatic differentiation underpins:

- backpropagation;
- gradient-based training;
- neural-network libraries;
- differentiable programming;
- physics-informed learning;
- meta-learning;
- differentiable optimization.


## 13. Decision Intelligence case — Differentiable policy simulator

Suppose a differentiable simulator maps two policy variables to a modeled outcome.


In [ ]:
def policy_objective(xs):
    health, education = xs
    return (
        4 * dual_log(health + 1)
        + 5 * dual_log(education + 1)
        - 0.02 * (health + education) ** 2
    )

policy = np.array([20.0, 25.0])
policy_value, policy_gradient = autodiff_gradient(
    policy_objective,
    policy,
)

pd.Series(
    policy_gradient,
    index=[
        "Health local sensitivity",
        "Education local sensitivity",
    ],
)


### Interpretation

The gradient measures local model sensitivity to the policy variables. It does not establish causal impact or ethical priority. Reliable decision use requires model validation, uncertainty analysis, constraints, distributional assessment, and human oversight.


## 14. Engineering notes

- Forward mode is efficient for few inputs and many outputs.
- Reverse mode is efficient for scalar outputs and many inputs.
- Gradient checking should use small test models, not full production models.
- Non-smooth points may cause valid methods to disagree.
- Randomness should be controlled during checks.
- Complex data pipelines can break differentiability.
- Correct derivatives do not guarantee a correct model.


## 15. Common errors

- Calling finite differences automatic differentiation.
- Forgetting to seed derivative components.
- Checking gradients with stochastic behavior enabled.
- Using a finite-difference step that is too large or too small.
- Ignoring non-differentiable operations.
- Assuming a passing gradient check validates the entire algorithm.


## 16. Exercises

### Level A
Explain symbolic, numerical, and automatic differentiation.

### Level B
Derive the dual-number product and quotient rules.

### Level C
Extend the `Dual` class with square root and hyperbolic tangent.

### Capstone
Build a differentiable policy objective, compute its gradient automatically, verify it numerically, and document all assumptions limiting decision use.


## 17. Key insight

Automatic differentiation turns the chain rule into an executable computational process. Gradient checking provides an independent numerical audit. Together, they make gradient-based optimization and AI systems more reliable and transparent.
